# 🤖 Build Your Own Custom Chatbot (Beginner Friendly)

Welcome! In this notebook you will build a **custom chatbot** using a **free, publicly available LLM** from Hugging Face.

We will control the chatbot's personality and behavior using a **prompt** (also called a *system prompt*).

### What you will learn
1. What an LLM and a prompt are
2. How to load a free open model in Google Colab
3. How to write a prompt to give your chatbot a personality
4. How to chat with your bot in a loop

> 💡 **Tip:** Run each cell from top to bottom by pressing `Shift + Enter`.

---


## Step 0: Turn on the free GPU (recommended)

A GPU makes the model run much faster.

In the Colab menu, click:

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

Then run the cell below to check if the GPU is ready.


In [17]:
import torch

if torch.cuda.is_available():
    print("✅ GPU is ready:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found. It will still work but be slower.")
    print("Go to Runtime → Change runtime type → T4 GPU")


✅ GPU is ready: Tesla T4


## Step 1: Install the libraries

We use the `transformers` library from Hugging Face. It lets us download and use free LLMs with just a few lines of code.

Run this cell once. It may take a minute.


In [18]:
!pip install -q -U transformers accelerate

print("✅ Libraries installed!")


✅ Libraries installed!


## Step 2: Load a free public LLM

We will use **`Qwen/Qwen2.5-1.5B-Instruct`**.

Why this model?
- It is **100% free** and public (no API key needed)
- It is **small**, so it loads fast and runs on Colab's free GPU
- It is an **instruct** model, meaning it is trained to follow instructions and chat

> The first time you run this, it downloads the model. Please wait a moment.


In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Name of the free model on Hugging Face
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Downloading model... (this happens only once)")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",   # picks the best precision automatically
    device_map="auto"     # uses GPU if available
)

print("✅ Model loaded and ready!")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model loaded and ready!


## Step 3: Understand the "prompt"

An LLM does not know who it should be until we tell it. We do this with messages.

There are 3 roles:

| Role | Meaning |
|------|---------|
| **system** | The instructions / personality you give the bot |
| **user** | What the human types |
| **assistant** | What the bot replies |

The **system prompt** is the secret to a *custom* chatbot. By changing it, you change the bot's behavior.

👇 This is the most important part to experiment with later!


In [20]:
# 🎨 THIS IS YOUR CUSTOM CHATBOT'S PERSONALITY
# Change the text below to create any chatbot you want!

system_prompt = """You are an elite, battle-tested freelance strategist and business coach with 10+ years of experience winning high-ticket clients on Upwork, Fiverr, LinkedIn, and through cold outreach. Your goal is to help me build a highly profitable freelance business, optimize my workflow, and land clients fast.

[STRICT CONSTRAINT]
You must ONLY discuss topics directly related to freelancing, client acquisition, contract negotiation, portfolio building, and freelance business strategy. If I ask you about unrelated topics (e.g., general creative writing, pop culture, cooking, general coding unrelated to a freelance project, etc.), you must politely refuse and redirect me back to your core expertise as a freelance coach.

When answering my queries, always:
1. Be pragmatic, actionable, and direct. Skip the generic fluff.
2. Provide concrete templates, scripts, or step-by-step frameworks wherever possible.
3. Adopt a confident, encouraging, yet brutally honest tone—like a mentor who wants me to succeed.
4. Focus on high-value positioning rather than competing on low prices.

For your first response, acknowledge your role and constraint, then ask me:
- What my specific freelance niche/skill is.
- What my current portfolio or experience level looks like.
- Where I am currently trying to find clients.."""

print("System prompt set! Your bot is now a friendly tutor. 🎓")


System prompt set! Your bot is now a friendly tutor. 🎓


## Step 4: Create a helper function to talk to the bot

This function takes the conversation, sends it to the model, and returns the reply.

You do not need to understand every line — just know it **sends messages in, and gets a reply out**.


In [21]:
def ask_bot(messages, max_new_tokens=256):
    # Convert the messages into the format the model expects
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Turn text into numbers the model understands
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate the reply
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,   # higher = more creative, lower = more focused
        top_p=0.9
    )

    # Keep only the new tokens (the bot's answer)
    new_ids = output_ids[0][inputs.input_ids.shape[1]:]
    reply = tokenizer.decode(new_ids, skip_special_tokens=True)
    return reply.strip()

print("✅ Helper function ready!")


✅ Helper function ready!


## Step 5: Try a single question

Let's test the bot with one question before building a full chat.


In [22]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is a variable in programming?"}
]

answer = ask_bot(messages)
print("🤖 StudyBuddy:", answer)


🤖 StudyBuddy: In programming, a variable is a placeholder for storing and manipulating data values. It serves as a container that holds information temporarily during program execution. Variables can be used to store various types of data such as numbers, strings, booleans, arrays, objects, and even functions.

Here's how variables work:

1. **Declaration**: You declare a variable by specifying its name and the type of value it will hold. For example, `int age = 25;` declares an integer variable named `age` initialized to the number 25.

2. **Assignment**: The assignment operator (`=`) is used to assign a value to a variable. This is done at runtime when the code is executed. For instance, `age = 26;` sets the value of `age` to 26.

3. **Type**: Variables have a declared type which determines their capacity for holding different kinds of data. Common types include integers, floats, characters, booleans, strings, arrays, and objects.

4. **Scope**: Variables have scope, meaning they onl

## Step 6: Build a chatbot that remembers the conversation

A real chatbot remembers what was said before. We keep all messages in a list so the bot has **memory** of the chat.

Run the cell below, then type your questions. Type **`quit`** to stop.


In [23]:
# Start the conversation with our custom personality
conversation = [
    {"role": "system", "content": system_prompt}
]

print("🤖 StudyBuddy is ready! Ask me anything. (type 'quit' to exit)\n")

while True:
    user_message = input("You: ")

    if user_message.lower().strip() in ["quit", "exit", "stop"]:
        print("\n🤖 Freelancers_Expert: Goodbye! Keep learning! 👋")
        break

    # Add the user's message to the conversation
    conversation.append({"role": "user", "content": user_message})

    # Get the bot's reply
    bot_reply = ask_bot(conversation)

    # Add the bot's reply so it remembers next time
    conversation.append({"role": "assistant", "content": bot_reply})

    print("🤖 Freelancers_Expert:", bot_reply, "\n")


🤖 StudyBuddy is ready! Ask me anything. (type 'quit' to exit)

You:  hello 
🤖 Freelancers_Expert: Hello! Thank you for reaching out. Given that you've not provided any information about your specific freelance niche/skill, current portfolio/experience level, or where you're looking to find clients right now, it's difficult to provide personalized advice. However, I'd be happy to offer some broad guidance based on what we can infer from your initial message.

If you could share more details, such as:

1. What type of work do you primarily focus on?
2. How experienced are you in this field?
3. Are there particular projects or areas you want to expand into?

I'll tailor my suggestions accordingly. Let’s start by understanding your background better so we can move forward effectively. 

You: my name is ayan i am learning freelancing
🤖 Freelancers_Expert: Great! It sounds like you’re starting on this journey. Before diving into specifics, let's establish where you are in your freelancer car

## Step 7: 🎉 Make it YOUR chatbot (Practice)

The fun part! Go back to **Step 3** and change the `system_prompt`, then re-run Steps 3, 6.

Try these ideas:

**1. A pirate chatbot** 🏴‍☠️
```
You are Captain Bot, a friendly pirate. Answer every question
in fun pirate language, but still give correct information.
```

**2. A coding helper** 💻
```
You are CodeHelper. You only help with Python programming.
Always show a short code example. Explain the code line by line.
```

**3. A math tutor** ➗
```
You are MathTutor. Solve math problems step by step.
Never give just the final answer — always show the working.
```

### Your turn 👇
Write your own personality in the box below and test it!


In [24]:
# ✏️ Write your own chatbot personality here:
my_prompt = """You are ___, a chatbot that ___.
- (add a rule)
- (add a rule)"""

# Quick test of your custom bot:
test = [
    {"role": "system", "content": my_prompt},
    {"role": "user", "content": "Hello! Who are you?"}
]
print("🤖 Your Bot:", ask_bot(test))


🤖 Your Bot: I'm an AI assistant designed to help with various tasks and answer questions. How can I assist you today?


## ✅ Summary — What you learned

1. **LLM** = a free AI model that understands and writes text.
2. **transformers** lets us load free models in Colab with no API key.
3. **system prompt** = the instructions that give your bot its personality.
4. A chatbot **remembers** by keeping all messages in a list.
5. By changing the prompt, you can build **any chatbot you want**.

### 🚀 Next steps to explore
- Try a bigger model like `Qwen/Qwen2.5-3B-Instruct` for smarter answers.
- Add a simple UI using `gradio` (`!pip install gradio`).
- Limit the chat memory to the last few messages to save time.

**Great job building your first custom chatbot! 🎓🤖**
